# P02 · Configuration space and A* search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/09_planning_learning/P02_configuration_space_and_a_search.ipynb)

**Track:** Planning and learning · **Skill axis (sheet):** Planning and learning · **Time:** about 60 min

**Skill Hub concepts:** Configuration space and A* search; Configuration space

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("P02")

## Map: plan for a point, not for a robot

**Configuration space (C-space):** the set of all robot configurations. A disc robot of radius $r$ translating in the
plane has a 2D C-space; growing every obstacle by $r$ (a Minkowski sum) lets you plan for a **point**. A rotating
rectangle needs a third dimension (heading); an arm's C-space is its joint angles.

**A\*** expands nodes in order of $f = g + h$, with $g$ the cost so far and $h$ a heuristic of the cost to go.

| Heuristic $h$ | Admissible ($h \le h^*$) | Result |
|---|---|---|
| $h = 0$ | yes | Dijkstra: optimal, expands everything closer than the goal |
| octile distance on an 8-connected grid | yes and consistent | optimal, far fewer expansions |
| $w \cdot$ octile, $w > 1$ (weighted A\*) | no | faster, cost at most $w$ times optimal |

Octile distance (8 moves, diagonal cost $\sqrt 2$): $h = \max(\Delta x, \Delta y) + (\sqrt 2 - 1)\min(\Delta x, \Delta y)$

Grid rules for this lab: 8-connected, straight cost 1, diagonal cost $\sqrt 2$, and a diagonal move is allowed only
when **both** orthogonal neighbors it passes are free (no corner cutting).

In [ ]:
import heapq
import numpy as np

def make_map():
    g = np.zeros((40, 40), bool)          # 0.1 m cells, indexed grid[x, y]
    g[0:28, 13] = True
    g[12:40, 27] = True
    g[14:20, 4:7] = True
    return g

## Task 1: inflate obstacles into C-space
Mark every cell within Euclidean distance `r` cells (inclusive, $dx^2 + dy^2 \le r^2$) of an obstacle.

In [ ]:
def inflate(grid, r):
    raise NotImplementedError  # TODO

lab.check("inflate", inflate)

## Task 2: A\* with the octile heuristic
Return `(path, expanded)`: `path` is the list of cells from start to goal (or `None` if there is none or start or goal
is blocked) and `expanded` counts nodes popped and closed.

In [ ]:
def octile(a, b):
    raise NotImplementedError  # TODO
    

def astar(grid, start, goal, h=octile):
    raise NotImplementedError  # TODO

lab.check("astar", astar, inflate, octile)

In [ ]:
try:
    import matplotlib.pyplot as plt
    raw = make_map(); cs = inflate(raw, 3)
    path, n = astar(cs, (2, 2), (37, 37))
    plt.imshow(cs.T.astype(int) + raw.T, origin="lower", cmap="Greys")
    plt.plot(*np.array(path).T, "r-"); plt.title(f"A*: {n} expansions"); plt.show()
except NotImplementedError:
    print("finish Tasks 1 and 2 first")

## Checkpoint: Math
**M1 (Apply).** Octile distance from cell (0, 0) to (5, 3)?

**M2 (Apply).** Robot radius 0.25 m on a 0.1 m grid: inflation radius in whole cells (round up)?

**M3 (Apply).** Cells marked when a single obstacle cell is inflated with $r = 3$?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** You multiply the octile heuristic by 2. What changes?
- a) Nothing: A\* is still optimal
- b) Usually fewer expansions, but the path can cost up to twice the optimum
- c) A\* can no longer find any path

**K2 (Understand).** Why plan in C-space instead of checking the robot's outline along each candidate path, and when
does a 2D inflated grid stop being enough?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `p02.h`).
3. In `roboprep/checks/p02.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()